# ThreatLens EMBER2024 Win32 Training (GPU first)

This trains a benign/malware LightGBM candidate on EMBER2024 Win32 PE vectors. It uses chronological train/validation/test separation, exact SHA-256 duplicate checks, memory-mapped feature arrays, and batched scoring.

Leakage controls: hashes, dates, AV detection metadata, family/type tags and labels are never model inputs; exact duplicate hashes are removed within training and from training if seen in test; validation follows fit rows chronologically; official test/challenge never affect fitting or threshold selection. Only the fixed EMBER v3 PE vector is modeled.

Default training sample: 150,000 rows spread over training chronology. GPU backends are attempted CUDA, OpenCL GPU, then CPU. GPU needs a GPU-enabled LightGBM build. After training, the paired model/schema/metrics are written to `backend/ml/artifacts/ember2024_candidate/`; the active model is not overwritten. Current backend expects a different 72-feature schema, so exact EMBER v3 extraction must be wired into inference before activating this candidate.

**Run the setup cell under section 1 first**. It installs all Python packages into the interpreter backing the currently selected Jupyter kernel, checks `thrember`, and verifies the EMBER v3 feature dimension before training. Works across Windows/macOS/Linux with a Python 3.10+ kernel, internet access, and package-install permission. Non-Python kernels, offline machines, and locked-down managed environments cannot install Python packages through a notebook.

Sources: [EMBER2024](https://github.com/FutureComputing4AI/EMBER2024), [LightGBM GPU setup](https://lightgbm.readthedocs.io/en/stable/GPU-Tutorial.html).

## 1. Unified environment setup (run this cell first)

This single setup cell uses `sys.executable` so packages go into the exact Python environment backing this notebook. It uses the official EMBER2024 source ZIP (no Git executable needed) and pins `signify==0.7.1`: upstream `thrember` currently has a reported Authenticode parsing issue with newer Signify versions. The installer uses normal CPU/GPU-capable LightGBM package selection; if GPU support is unavailable, the training section falls back to CPU.

Run this cell once, wait for the `Setup complete` message, and then run the remaining cells. It is safe to run again if packages are already installed. If the selected environment is read-only, select a Python kernel whose environment allows package installation. Python 3.10 or newer is required by `thrember`.


In [ ]:
import importlib
import importlib.metadata as metadata
import os
import subprocess
import sys
import tempfile
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError(
        f"This notebook requires a Python 3.10+ Jupyter kernel; the selected kernel is {sys.version.split()[0]} "
        f"({sys.executable}). In Jupyter, switch to an installed Python 3.10+ kernel and rerun this cell."
    )

print("Installing into the active Jupyter kernel:", sys.executable)

# Some minimal Python distributions omit pip. Bootstrap it when ensurepip is available.
pip_check = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True)
if pip_check.returncode != 0:
    ensure = subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], text=True)
    if ensure.returncode != 0:
        raise RuntimeError(
            "This Python kernel has no usable pip and ensurepip could not install it. "
            "Select a standard Python 3.10+ kernel with pip enabled, then rerun this cell."
        )

# One requirements transaction lets pip resolve compatible wheels together.
# Pin Signify because upstream thrember currently slices a certificate collection
# that newer Signify releases return as a non-subscriptable CertificateStore.
requirements = """numpy<3
pandas>=2.2
scikit-learn>=1.5.1
lightgbm>=4.6,<5
joblib>=1.4
matplotlib>=3.10.3
huggingface_hub>=0.32.4,<1.0
ipykernel
pefile>=2024.8.26
polars>=1.8.2
tqdm>=4.66.5
signify==0.7.1
thrember @ https://github.com/FutureComputing4AI/EMBER2024/archive/refs/heads/main.zip
"""
req_path = None
try:
    with tempfile.NamedTemporaryFile(mode="w", suffix="-threatlens-ember-requirements.txt", encoding="utf-8", delete=False) as f:
        f.write(requirements)
        req_path = f.name
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--prefer-binary", "-r", req_path],
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Package installation failed. Check the pip error printed above. Common causes are no internet/proxy access, "
            "a read-only Python environment, or a Python version with no compatible package wheels. "
            "Fix that environment issue and rerun this setup cell."
        )
finally:
    if req_path and os.path.exists(req_path):
        os.remove(req_path)

importlib.invalidate_caches()
import numpy, pandas, sklearn, lightgbm, joblib, matplotlib, huggingface_hub, thrember
from thrember.features import PEFeatureExtractor
feature_dim = PEFeatureExtractor().dim
if feature_dim != 2568:
    raise RuntimeError(f"Unexpected thrember feature schema: expected 2568 dimensions, got {feature_dim}.")
print("Setup complete. Package versions:", {
    name: metadata.version(name)
    for name in ("numpy", "pandas", "scikit-learn", "lightgbm", "thrember", "signify")
})
print("EMBER v3 feature dimension:", feature_dim)


In [ ]:
from pathlib import Path
import os, sys, json, platform, shutil
from datetime import datetime, timezone
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, lightgbm as lgb, thrember
from thrember.features import PEFeatureExtractor
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
 f1_score, roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix,
 roc_curve, precision_recall_curve)

SEED=42
MAX_TRAIN_ROWS=150_000       # raise only if RAM/VRAM permits
VALIDATION_FRACTION=0.15     # latest part of training timeline
TARGET_VALIDATION_FPR=0.01
EVAL_BATCH_SIZE=20_000
PREFER_GPU=True

def find_root(start=Path.cwd()):
    for p in (start,*start.parents):
        if (p/"backend"/"ml"/"artifacts").is_dir(): return p
    raise FileNotFoundError("Start Jupyter from the ThreatLens project folder.")
ROOT=find_root()
DATA=ROOT/"data"/"ember2024_win32"
OUT=ROOT/"backend"/"ml"/"artifacts"/"ember2024_candidate"
DATA.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
D=int(PEFeatureExtractor().dim)
assert D==2568,f"Unexpected EMBER v3 feature dimension {D}"
CAT=[2,3,4,5,6,701,702]
print("Root:",ROOT,"Data:",DATA,"Output:",OUT)
print("Features:",D,"LightGBM:",lgb.__version__,"Python:",sys.version.split()[0])
print("Free disk GiB:",round(shutil.disk_usage(DATA).free/1024**3,1))

## 2. Download Win32 and vectorize with EMBER v3

The official downloader supports `file_type="Win32"`; official Win32 train/test archives are about 23.7 GB and 4.9 GB. Leave extra space for vector files. We avoid the upstream convenience reader because it copies full arrays into RAM.

In [ ]:
train_files=sorted(DATA.glob("*train*.jsonl")); test_files=sorted(DATA.glob("*test*.jsonl"))
if not train_files or not test_files:
    old=Path.cwd()
    try: thrember.download_dataset(str(DATA),file_type="Win32",split="all")
    finally: os.chdir(old)
train_files=sorted(DATA.glob("*train*.jsonl")); test_files=sorted(DATA.glob("*test*.jsonl"))
challenge_files=sorted(DATA.glob("*challenge*.jsonl"))
assert train_files and test_files,"Train/test JSONL shards missing."

def vectors_ok(s):
    x,y=DATA/f"X_{s}.dat",DATA/f"y_{s}.dat"
    return x.is_file() and y.is_file() and x.stat().st_size>0 and y.stat().st_size>0
if not vectors_ok("train") or not vectors_ok("test") or (challenge_files and not vectors_ok("challenge")):
    print("Vectorizing; this can take hours on a laptop.")
    thrember.create_vectorized_features(str(DATA),label_type="label")
def rows(s):
    x,y=DATA/f"X_{s}.dat",DATA/f"y_{s}.dat"
    if not x.exists() or not y.exists(): return None
    rb=D*4
    if x.stat().st_size%rb or y.stat().st_size%4: raise ValueError(f"Invalid {s} vector sizes")
    nx,ny=x.stat().st_size//rb,y.stat().st_size//4
    if nx!=ny: raise ValueError(f"{s} X/y row mismatch: {nx} != {ny}")
    return int(nx)
NTR,NTE,NCH=rows("train"),rows("test"),rows("challenge")
print({"train":NTR,"test":NTE,"challenge":NCH})

## 3. Hash audit and chronological train/validation sample

Scan raw metadata in sorted shard order (same order as vectorization). Keep earliest train occurrence; remove train rows with a SHA-256 also in test. Hashes are used only to audit duplicates, never as features. Choose evenly spaced rows without looking at labels; reserve the latest 15% for validation. No oversampling or balancing.

In [ ]:
def read_hashes(subset,n):
    paths=sorted(DATA.glob(f"*{subset}*.jsonl"))
    if not paths: raise FileNotFoundError(f"No {subset} JSONL files")
    out=np.empty(n,dtype="S64"); i=0
    for path in paths:
        with path.open(encoding="utf-8") as f:
            for line in f:
                if i>=n: raise ValueError("Metadata rows exceed feature rows")
                h=json.loads(line).get("sha256")
                out[i]=str(h).strip().lower().encode("ascii","ignore")[:64] if h else b""
                i+=1
    if i!=n: raise ValueError(f"{subset}: metadata {i}, features {n}")
    return out
th=read_hashes("train",NTR); eh=read_hashes("test",NTE)
test_hashes={h for h in eh if h}; seen=set(); keep=np.zeros(NTR,dtype=bool)
dups=overlap=missing=0
for i,h in enumerate(th):
    if not h: keep[i]=True; missing+=1
    elif h in test_hashes: overlap+=1
    elif h in seen: dups+=1
    else: seen.add(h); keep[i]=True
train_indices=np.flatnonzero(keep)
print({"train_unique_kept":len(train_indices),"train_duplicates_removed":dups,
 "train_test_overlaps_removed":overlap,"missing_hashes_kept":missing})
assert len(train_indices)>0

Xtr=np.memmap(DATA/"X_train.dat",dtype=np.float32,mode="r",shape=(NTR,D))
ytr=np.memmap(DATA/"y_train.dat",dtype=np.int32,mode="r",shape=(NTR,))
Xte=np.memmap(DATA/"X_test.dat",dtype=np.float32,mode="r",shape=(NTE,D))
yte_mm=np.memmap(DATA/"y_test.dat",dtype=np.int32,mode="r",shape=(NTE,))
n=min(MAX_TRAIN_ROWS,len(train_indices))
sample=train_indices[np.linspace(0,len(train_indices)-1,n,dtype=np.int64)]
sample=sample[np.isin(np.asarray(ytr[sample],dtype=np.int32),[0,1])]
split=int((1-VALIDATION_FRACTION)*len(sample))
if split<1000 or len(sample)-split<200: raise ValueError("Too few labeled rows; increase MAX_TRAIN_ROWS.")
fit_idx,val_idx=sample[:split],sample[split:]
yf=np.asarray(ytr[fit_idx],dtype=np.int32); yv=np.asarray(ytr[val_idx],dtype=np.int32)
Xf=np.asarray(Xtr[fit_idx],dtype=np.float32); Xv=np.asarray(Xtr[val_idx],dtype=np.float32)
assert fit_idx.max()<val_idx.min() and set(np.unique(yf))=={0,1} and set(np.unique(yv))=={0,1}
print("Fit:",Xf.shape,np.bincount(yf),"Val:",Xv.shape,np.bincount(yv))
print("Selected matrices GiB:",round((Xf.nbytes+Xv.nbytes)/1024**3,2))

## 4. GPU-first LightGBM training

Moderate tree complexity and regularization with early stopping help limit overfitting. Early stopping uses validation only. CUDA is tried first, then OpenCL GPU, then CPU; each fallback starts with a fresh estimator.

In [ ]:
params=dict(objective="binary",n_estimators=700,learning_rate=0.05,num_leaves=63,
 max_depth=-1,min_child_samples=100,subsample=0.8,subsample_freq=1,
 colsample_bytree=0.8,reg_lambda=2.0,random_state=SEED,
 n_jobs=min(8,os.cpu_count() or 4),verbosity=-1,max_bin=63)
callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(100)]
model=DEVICE=None; gpu_errors=[]
for device in ((["cuda","gpu"] if PREFER_GPU else [])+["cpu"]):
    try:
        print("Trying:",device)
        candidate=lgb.LGBMClassifier(**params,device_type=device)
        candidate.fit(Xf,yf,eval_set=[(Xv,yv)],eval_metric=["auc","binary_logloss"],
                      categorical_feature=CAT,callbacks=callbacks)
        model,DEVICE=candidate,device; break
    except Exception as exc:
        gpu_errors.append(f"{device}: {type(exc).__name__}: {str(exc)[:400]}")
        print("Fallback:",gpu_errors[-1])
        if device=="cpu": raise
if model is None: raise RuntimeError("No LightGBM backend trained.")
print("Device:",DEVICE,"Best iteration:",model.best_iteration_,"GPU issues:",gpu_errors)

## 5. Validation threshold, final test, challenge

Select an operating threshold on benign validation rows for <=1% validation FPR, then freeze it. The official later test is scored in batches once. Do not tune from test/challenge metrics. Challenge is malware-focused, so report detection rate only. Raw model scores are not necessarily calibrated probabilities.

In [ ]:
sv=model.predict_proba(Xv,num_iteration=model.best_iteration_)[:,1]
def threshold_at_fpr(y,s,target=0.01):
    benign=np.sort(np.asarray(s)[np.asarray(y)==0])
    allowed=int(np.floor(target*len(benign)))
    return float(np.nextafter(benign[max(0,len(benign)-allowed-1)],np.inf))
THRESHOLD=threshold_at_fpr(yv,sv,TARGET_VALIDATION_FPR)
print("Validation:",{"threshold":THRESHOLD,"auc":float(roc_auc_score(yv,sv)),
 "average_precision":float(average_precision_score(yv,sv)),
 "brier_raw_score":float(brier_score_loss(yv,sv)),
 "fpr":float(np.mean(sv[yv==0]>=THRESHOLD))})

def scores_batched(est,X,n):
    out=np.empty(n,dtype=np.float32)
    for a in range(0,n,EVAL_BATCH_SIZE):
        b=min(a+EVAL_BATCH_SIZE,n)
        out[a:b]=est.predict_proba(np.asarray(X[a:b],dtype=np.float32),
                                   num_iteration=est.best_iteration_)[:,1]
    return out
def threshold_metrics(y,s,t):
    pred=np.asarray(s)>=t; tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"threshold":float(t),"accuracy":float(accuracy_score(y,pred)),
     "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
     "precision_malware":float(precision_score(y,pred,zero_division=0)),
     "recall_malware":float(recall_score(y,pred,zero_division=0)),
     "f1_malware":float(f1_score(y,pred,zero_division=0)),
     "fpr":float(fp/(fp+tn)) if fp+tn else None,"fnr":float(fn/(fn+tp)) if fn+tp else None,
     "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}
stest=scores_batched(model,Xte,NTE); ytall=np.asarray(yte_mm,dtype=np.int32)
mask=np.isin(ytall,[0,1]); yt,st=ytall[mask],stest[mask]
assert set(np.unique(yt))=={0,1}
test_report={"rows":int(len(yt)),"roc_auc":float(roc_auc_score(yt,st)),
 "average_precision":float(average_precision_score(yt,st)),
 "brier_raw_score_not_calibrated":float(brier_score_loss(yt,st)),
 "threshold_0_5":threshold_metrics(yt,st,0.5),
 "validation_threshold":threshold_metrics(yt,st,THRESHOLD)}
print("Final test:",json.dumps(test_report,indent=2))

fig,ax=plt.subplots(1,2,figsize=(12,4.5))
fpr,tpr,_=roc_curve(yt,st); ax[0].plot(fpr,tpr,label=f"AUC {test_report['roc_auc']:.4f}")
ax[0].plot([0,1],[0,1],"--",color="gray"); ax[0].axvline(TARGET_VALIDATION_FPR,ls=":",color="orange")
ax[0].set(title="Untouched test ROC",xlabel="FPR",ylabel="TPR"); ax[0].legend(); ax[0].grid(alpha=.25)
p,r,_=precision_recall_curve(yt,st); ax[1].plot(r,p,label=f"AP {test_report['average_precision']:.4f}")
ax[1].set(title="Test precision-recall",xlabel="Recall",ylabel="Precision"); ax[1].legend(); ax[1].grid(alpha=.25)
plt.tight_layout(); plt.show()

challenge_report=None
if NCH:
    Xch=np.memmap(DATA/"X_challenge.dat",dtype=np.float32,mode="r",shape=(NCH,D))
    ych=np.memmap(DATA/"y_challenge.dat",dtype=np.int32,mode="r",shape=(NCH,))
    sch=scores_batched(model,Xch,NCH); yc=np.asarray(ych,dtype=np.int32); mal=yc==1
    if mal.any():
        challenge_report={"n_malware":int(mal.sum()),
         "detection_rate_at_validation_threshold":float(np.mean(sch[mal]>=THRESHOLD)),
         "median_score":float(np.median(sch[mal])),"p95_score":float(np.quantile(sch[mal],.95))}
print("Challenge detection-only:",challenge_report)

## 6. Export paired candidate model and schema into ThreatLens

Do not overwrite the active model. This candidate expects EMBER v3's 2,568-element ordered vector, while the current backend extractor emits a different 72-field schema. Backend changes must use the exact same extractor and reject invalid/non-PE files safely before this can be activated.

In [ ]:
names=[f"ember_v3_{i:04d}" for i in range(D)]
schema={"schema_name":"ember2024_win32_pe_v3","extractor":"thrember.PEFeatureExtractor",
 "feature_count":D,"feature_names":names,"categorical_feature_indices":CAT,
 "label_mapping":{"0":"benign","1":"malware"}}
manifest={"model_name":"ThreatLens EMBER2024 Win32 LightGBM candidate",
 "dataset":"EMBER2024","source":"https://github.com/FutureComputing4AI/EMBER2024",
 "feature_version":"EMBER v3 / thrember","feature_count":D,
 "train_rows":int(NTR),"unique_train_rows":int(len(train_indices)),
 "sample_rows":int(len(sample)),"fit_rows":int(len(yf)),"validation_rows":int(len(yv)),
 "test_rows":int(len(yt)),"device":DEVICE,"gpu_attempt_errors":gpu_errors,"seed":SEED,
 "validation_target_fpr":TARGET_VALIDATION_FPR,"threshold":THRESHOLD,
 "best_iteration":int(model.best_iteration_ or params["n_estimators"]),
 "validation_auc":float(roc_auc_score(yv,sv)),"test_metrics":test_report,
 "challenge_metrics":challenge_report,"trained_utc":datetime.now(timezone.utc).isoformat(),
 "limitations":["Benchmark metrics do not guarantee operational accuracy.",
 "Backend must use exact EMBER v3 extraction and order before live inference.",
 "Do not tune on final test or challenge results."]}
joblib.dump(model,OUT/"ember2024_lightgbm.joblib",compress=3)
model.booster_.save_model(str(OUT/"ember2024_lightgbm.txt"))
(OUT/"feature_schema.json").write_text(json.dumps(schema,indent=2),encoding="utf-8")
(OUT/"metrics.json").write_text(json.dumps(test_report,indent=2),encoding="utf-8")
(OUT/"training_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
pd.DataFrame({"index":np.arange(D),"feature_name":names,
 "gain":model.booster_.feature_importance(importance_type="gain"),
 "split_count":model.booster_.feature_importance(importance_type="split")
}).sort_values("gain",ascending=False).to_csv(OUT/"feature_importance.csv",index=False)
for f in sorted(OUT.iterdir()):
    if f.is_file(): print(f.name,round(f.stat().st_size/1024**2,2),"MiB")
print("Candidate saved; active artifact untouched.")

## Leakage and live-use checklist

- Keep the official future test untouched. If you change the model after reviewing its results, obtain a new final holdout.
- Training/validation separation is chronological; sample selection does not use labels.
- Exact duplicate hashes in training and train/test are filtered from training.
- Model inputs are only the fixed EMBER v3 vector, never hashes, dates, AV detections, labels, or family tags.
- Challenge reports detection rate only and does not set the threshold.
- Benchmark metrics cannot guarantee every live result. Evaluate on recent independently labeled operational files.
- Until backend inference emits the exact same EMBER v3 vector from uploaded PE bytes, leave this candidate inactive.